## Basic trying things (skip)

In [ ]:
import kdsource.api as kds
import numpy as np

In [ ]:
help(kds.write_mcpl)

In [ ]:
file_path = "my_data.mcpl.gz"

empty_10 = np.zeros(10)
full_10 = np.ones(10)
kds.write_mcpl(filename=file_path, ekin=full_10, x=empty_10, y=empty_10, z=empty_10, ux=full_10, uy=empty_10, uz=empty_10, time=empty_10, weight=full_10, pdgcode=2112)

In [ ]:
import mcpl
import pandas as pd

myfile=mcpl.MCPLFile(file_path,blocklength=2)
datas = []
for i,p in enumerate(myfile.particle_blocks):
    print("block no.", i)
    data = pd.DataFrame({
        'Kf': p.pdgcode,  # Particle PDG code
        'x': p.x,  # x-position
        'y': p.y,  # y-position
        'z': p.z,  # z-position
        'Vx': p.ux,  # Velocity in x-direction
        'Vy': p.uy,  # Velocity in y-direction
        'Vz': p.uz,  # Velocity in z-direction
        'e': p.ekin,  # Kinetic energy
        'Wt': p.weight  # Weight of the particle
    })
    datas.append(data)
data = pd.concat(datas, ignore_index=True)
data

In [ ]:
data.dtypes

In [ ]:
myfile=mcpl.MCPLFile("./examples/uniform/output.mcpl.gz",blocklength=100)
datas = []
for i,p in enumerate(myfile.particle_blocks):
    if i==1: break
    data = pd.DataFrame({
        'Kf': p.pdgcode,  # Particle PDG code
        'x': p.x,  # x-position
        'y': p.y,  # y-position
        'z': p.z,  # z-position
        'Vx': p.ux,  # Velocity in x-direction
        'Vy': p.uy,  # Velocity in y-direction
        'Vz': p.uz,  # Velocity in z-direction
        'e': p.ekin,  # Kinetic energy
        'Wt': p.weight  # Weight of the particle
    })
    datas.append(data)
data = pd.concat(datas, ignore_index=True)
data

In [ ]:
kds.write_mcpl(filename=file_path, ekin=data["e"], x=data["x"], y=data["y"], z=data["z"],
                ux=data["Vx"], uy=data["Vy"], uz=data["Vz"], time=np.zeros_like(data["Kf"]), weight=data["Wt"], pdgcode=data["Kf"])

In [ ]:
import kdsource.api as kds  # Importing the KDSource Python API

In [ ]:
kds.Geometry?

In [ ]:
kds.Metric?

In [ ]:
kds.geom._metrics

In [ ]:
my_energy = kds.geom.Energy()
#my_lethargy = kds.geom.Lethargy()

In [ ]:
type(my_energy)

In [ ]:
for k,v in my_energy.__dict__.items():
    print(k, str(v))

In [ ]:
# for k,v in my_lethargy.__dict__.items():
#     print(k, str(v))

In [ ]:
my_position = kds.geom.Vol()
my_time = kds.geom.Time()
my_directions = kds.geom.Isotrop()

In [ ]:
my_geometry = kds.Geometry([
    my_energy, my_time, my_position, my_directions
])

In [ ]:
for k,v in my_geometry.__dict__.items():
    print(k, str(v))

In [ ]:
kds.KDSource?

In [ ]:
import kdsource

In [ ]:
kdsource?

In [ ]:
kds.kde.bw_methods

In [ ]:
# Step 1: Load your original MCPL particle track list file
my_input_file_path = "./examples/uniform/output.mcpl.gz"

my_input_plist = kds.PList(filename=my_input_file_path,
                           readformat="mcpl",
                           pt="n",
                           trasl=None,
                           rot=None,
                           switch_x2z=False,
                           set_params=True)


In [ ]:
vars(my_input_plist)

In [ ]:
my_source.fitted

In [ ]:
my_source = kds.KDSource(
    plist=my_input_plist, geom=my_geometry, bw="mlcv", J=1.0, kernel="gaussian"
)

In [ ]:
my_source.fitted

In [ ]:
[attr for attr in dir(my_source) if attr[0:2]!="__"]

In [ ]:
vars(my_source.plist)

In [ ]:
my_source.fitted, my_source.bw_method

In [ ]:
my_source.fit(N=10000)

In [ ]:
vars(my_source.geom)

In [ ]:
my_source.kde.fit?

In [ ]:
my_source.fitted

In [ ]:
import numpy as np
my_source.plot_t(np.array([-1,0,1]))

In [ ]:
my_source.kde.data

In [ ]:
my_source.plot_E(grid_E=np.array([0,1e-8,2e-8,3e-8]))

In [ ]:
my_source.plot_integr(var="x", grid=np.arange(-5,5,0.2));

In [ ]:
my_source.save()

In [ ]:
kds.resample_to_mcpl("_source.xml", "generated_from_source.mcpl.gz", nout=100)

In [ ]:
import mcpl
import pandas as pd

gen_file=mcpl.MCPLFile("generated_from_source.mcpl.gz")
datas = []
for i,p in enumerate(gen_file.particle_blocks):
    print("block no.", i)
    data = pd.DataFrame({
        'Kf': p.pdgcode,  # Particle PDG code
        'x': p.x,  # x-position
        'y': p.y,  # y-position
        'z': p.z,  # z-position
        'Vx': p.ux,  # Velocity in x-direction
        'Vy': p.uy,  # Velocity in y-direction
        'Vz': p.uz,  # Velocity in z-direction
        'e': p.ekin,  # Kinetic energy
        'Wt': p.weight  # Weight of the particle
    })
    datas.append(data)
data = pd.concat(datas, ignore_index=True)
data

In [ ]:
my_input_plist.N

In [ ]:
kdsource.plist.pt2pdg?

# method definitions (skip)
read_data, resample_df, resample_df_gen, generate_data, plot_data_row

In [1]:
import kdsource.api as kds
import mcpl
import pandas as pd
from pathlib import Path
import tempfile
import numpy as np


def is_constant_column(column):
    return np.all(column == column[0], axis=0)


def read_data(file_path, n=-1):
    """
    Read up to n particles from the mcpl file. n==-1 means read all.
    """
    blocklength = 10000
    gen_file = mcpl.MCPLFile(file_path, blocklength=blocklength)
    datas = []
    for i, p in enumerate(gen_file.particle_blocks):
        data = pd.DataFrame(
            {
                "ekin": p.ekin,  # Kinetic energy
                "x": p.x,  # x-position
                "y": p.y,  # y-position
                "z": p.z,  # z-position
                "ux": p.ux,  # Velocity in x-direction
                "uy": p.uy,  # Velocity in y-direction
                "uz": p.uz,  # Velocity in z-direction
                "time": p.time,
                "weight": p.weight,  # Weight of the particle
                "pdgcode": p.pdgcode,  # Particle PDG code
            }
        )
        datas.append(data)
        if n != -1 and (i + 1) * blocklength >= n:
            break
    data = pd.concat(datas, ignore_index=True)
    if n != -1 and len(data) > n:
        data = data[:n]
    return data

def resample_df(
    source,
    dest,
    nout,
    rng=None,
    force=False,
    no_df=False,
    del_dest=False,
    del_source=False,
):
    """
    Generates data at `dest` and by default returns it as a pandas dataframe.
    - source: source model .xml file (which contains the original data path that is required for sampling)
    - dest: a file path with .mcpl.gz ending
    - nout: amount of particles generated
    - rng: rng parameter
    - force: whether `dest` file may be overwritten if it already exists
    - no_df: whether you do NOT want to receive a dataframe for some reason
    - del_dest: whether to delete file at `dest` in the end
    - del_source: whether or not to delete the source .xml file in the end

    Returns
    - pandas dataframe, or `dest`, or None
    """
    # print("Amount of particles to generate:", str(nout))
    kds.resample_to_mcpl(
        kds_sourcefile=source,
        destination_mcpl=dest,
        nout=nout,
        rng=rng,
        force=force,
    )
    # print("Data generated at", dest)
    if not no_df:
        data = read_data(dest)
    if del_dest:
        Path(dest).unlink()
    if del_source:
        Path(source).unlink()
    if not no_df:
        return data
    if not del_dest:
        return dest
    return None

def resample_df_gen(source, dest, nout, rng=None, del_dest=False, del_source=False):
    """
    A generator for dataframes via the resample method using an intermediate mcpl.gz file at `dest`. ATTENTION: Overwrites `dest` in each iteration!
    - source: source model .xml file (which contains the original data path that is required for sampling)
    - dest: a file path with .mcpl.gz ending
    - nout: amount of particles per dataframe
    - rng: If you specify it, it will be used for all the generated data. If None, uses new random rng for each iteration.
    - del_dest: whether or not to delete the most recent intermediate mcpl.gz file on the disk
    - del_source: whether or not to delete the source .xml file in the end (after the generator has stopped)
    """
    try:
        while True:
            yield resample_df(
                source=source,
                dest=dest,
                nout=nout,
                rng=rng,
                force=True,
                no_df=False,
                del_dest=del_dest,
                del_source=False,
            )
    finally:
        if del_source:
            Path(source).unlink()




def generate_data(
    inputFile,
    output_path=None,
    overwrite_output=False,
    n_gen=1_000,
    pt="n",
    n_fit=-1,
    kernel="gaussian",
    bw="silv",
    pCurrent=1.0,
    model_path=None,
    overwrite_model=False,
    generator=False,
):
    """
    Generates data similar to inputFile using kdsource, which uses KDE (Kernel Density Estimation) modeling.

    Parameters
    ---
    - inputFile: file path to input data; requires ending `.mcpl` or `.mcpl.gz`
    - output_path: file path for generated data (optional, see "Returns" section); file ending will be `.mcpl.gz`
    - overwrite_output: whether already existing file at output_path may be overwritten
    - n_gen: amount of particles to generate
    - pt: particle type ("n", "p", or "e") of generated particles and of input particles used for fitting the KDE model
    - n_fit: amount of input particles (with type `pt`) used for fitting of the source model (-1 means all)
    - kernel: kernel shape ("gaussian", "box", or "epa" (Epanechnikov))
    - bw: method for finding kernel bandwidth ("silv", "knn" or "mlcv")
    - pCurrent: total source particle current
    - model_path: where to save the model xml file (None makes it temporary)
    - overwrite_model: whether already existing file at model_path may be overwritten
    - generator: whether you want a generator for dataframes instead of one dataframe. `n_gen` will be the amount per yield.

    Returns
    ---
    - If output_path is specified: None. (The data will be saved to the file.)
    - If output_path is None: Pandas dataframe. Or, if `generator` is True: Generator of pandas dataframes.
    """

    inputFile = str(Path(inputFile).absolute())

    return_a_df = False
    output_is_temp = False
    model_is_temp = False

    if output_path is not None:
        output_path = Path(output_path).absolute().with_suffix(".mcpl.gz")
        if output_path.exists() and not overwrite_output:
            raise ValueError(
                "Output file already exists and overwriting is not allowed"
            )
        output_path = str(output_path)
    else:
        return_a_df = True
        output_path = str(Path(tempfile.gettempdir()) / "brightify_kdsource.mcpl.gz")
        output_is_temp = True

    if model_path is not None:
        model_path = Path(model_path).absolute().with_suffix(".xml")
        if model_path.exists() and not overwrite_model:
            raise ValueError("Model file already exists and overwriting is not allowed")
        model_path = str(model_path)
    else:
        model_path = str(Path(tempfile.gettempdir()) / "brightify_kdsource.xml")
        model_is_temp = True

    my_input_plist = kds.PList(
        filename=inputFile,
        readformat="mcpl",
        pt=pt,
        trasl=None,
        rot=None,
        switch_x2z=False,
        set_params=True,
    )
    # print(vars(my_input_plist))

    my_energy = kds.geom.Energy()
    my_time = kds.geom.Time()
    my_position = kds.geom.Vol()
    my_directions = kds.geom.Isotrop()
    my_geometry = kds.Geometry([my_energy, my_time, my_position, my_directions])
    # print(vars(my_geometry))

    my_source = kds.KDSource(
        plist=my_input_plist, geom=my_geometry, bw=bw, J=pCurrent, kernel=kernel
    )
    # print(vars(my_source))

    if n_fit == -1 or n_fit > my_input_plist.N:
        n_fit = my_input_plist.N
    # print("Amount of particles for fitting:", n_fit)

    my_source.fit(N=n_fit)

    model_path = str(Path(my_source.save(xmlfilename=model_path)).absolute())
    # if not model_is_temp:
    #     print("Source model saved to", model_path)
    # else:
    #     print("Temporary source model saved to", model_path)



    if not return_a_df:
        kds.resample_to_mcpl(
            kds_sourcefile=model_path,
            destination_mcpl=output_path,
            nout=n_gen,
            rng=None,
            force=True,
        )
        if model_is_temp:
            Path(model_path).unlink()
        return output_path
    else:
        if not generator:
            return resample_df(
                source=model_path,
                dest=output_path,
                nout=n_gen,
                rng=None,
                force=True,
                no_df=False,
                del_dest=output_is_temp,
                del_source=model_is_temp,
            )
        else:
            return resample_df_gen(
                source=model_path,
                dest=output_path,
                nout=n_gen,
                rng=None,
                del_dest=output_is_temp,
                del_source=model_is_temp,
            )

In [2]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def plot_data_row(datas, rows=["x", "ux", "ekin", "weight"]):
    colors = list(mcolors.TABLEAU_COLORS.values())
    for j, row in enumerate(rows):
        color = colors[j]
        for i, data in enumerate(datas):
            plt.hist(data[row], density=True, bins=20, histtype="step", label=row+" "+str(i), color=color, linestyle=(0,(i%2+1,i*2+1)))
    plt.legend()
    plt.show()

# method examples (skip)

In [ ]:
example_path = "./examples/uniform/output.mcpl.gz"

In [ ]:
data1 = generate_data(inputFile=example_path, n_gen=100_000, n_fit=1_000_000, bw="silv")

In [ ]:
data2 = read_data(example_path, 100_000)

In [ ]:
gen = generate_data(
    inputFile=example_path,
    output_path=None,
    overwrite_output=False,
    n_gen=100,
    pt="n",
    n_fit=100_000,
    kernel="gaussian",
    bw="silv",
    pCurrent=1.0,
    model_path=None,
    overwrite_model=False,
    generator=True,
)

In [ ]:
data3=next(gen)

In [ ]:
gen.close()

In [ ]:
data3=next(gen)

In [ ]:
import numpy as np

In [ ]:
for d in [data1, data2, data3]:
    d["V_norm"] = np.sqrt((d.Vx*d.Vx + d.Vy*d.Vy + d.Vz+d.Vz))

In [ ]:
data1.describe()

In [ ]:
data2.describe()

In [ ]:
data3.describe()

In [ ]:
plot_data_row(datas=[data1, data2, data3], rows=["e", "x", "Vx"])

# Exercise 26.10. - 30.10.

## file paths and manual inspection (skip)

In [3]:
example_path_1 = "./examples/Brightify Fig 6/Fig6a/output.mcpl.gz"  # const: ekin=2.5e-08, weight=1; almost const: z=0.06; n = 20_139_128
example_path_2 = "./examples/Brightify Fig 6/Fig6b/output.mcpl.gz"  # const: ekin=2.5e-08, weight=1; almost const: z=0.06; n = 21_580_959 (SAME AS 5)
example_path_3 = "./examples/Brightify Fig 6/Fig6c/output.mcpl.gz"  # const: ekin=2.5e-08, weight=1; almost const: z=0.06; n = 11_002_813
example_path_4 = "./examples/mcpl_different_normal/dump_final.mcpl.gz"  # almost const: x=13.718; n = 2_738_810
example_path_5 = "./examples/uniform/output.mcpl.gz"  # const: ekin=2.5e-08, weight=1; almost const: z=0.06; n = 21_580_959 (SAME AS 2)
example_path_6 = "./examples/resampled.mcpl.gz"  # const: z=286.915, weight=1; n = 2_000_000

example_paths = {
    1: example_path_1,
    2: example_path_2,
    3: example_path_3,
    4: example_path_4,
    5: example_path_5, # is the same as 2
    6: example_path_6,
}

In [ ]:
# !pymcpltool --stats "$example_path_4"

In [ ]:
# !pymcpltool --stats --gui "$example_path_6" # it did not work inside the notebook here, but from terminal (because of WSL)

## calculate silverman & mlcv and gaussian & epa, for 2 different data (skip)

### declare files, plist parameters and geometries

In [ ]:
# plist parameters:
pt = {index:"n" for index in example_paths.keys()}
trasl = {}
rot = {}
switch_x2z = {4: True} # swaps x to z, y to x, z to y

geometries = { # one per file
    1: kds.geom.Geometry([kds.geom.SurfXY(z=0.06)]), # omitting all dimensions except x and y position
    2: kds.geom.Geometry([kds.geom.SurfXY(z=0.06)]),
    3: kds.geom.Geometry([kds.geom.SurfXY(z=0.06)]),
    4: kds.geom.GeomFlat(z=13.718, E0=8.18e-08),
    5: kds.geom.Geometry([kds.geom.SurfXY(z=0.06)]),
    6: kds.geom.GeomFlat(z=286.915, E0=10),
}

### choose what to calculate

In [ ]:
bw_kernel_combinations = {  # choose from these by referencing the number/index below
    1: ("silv", "gaussian"),
    2: ("silv", "epa"),
    3: ("mlcv", "gaussian"),
    4: ("mlcv", "epa"),
}

calculate_these = [  # format: (file_index, bw_kernel_index, n_fit)
    # where n_fit is the number of particles (after "pt" filtering) to use for the fit.
    # type -1 for n_fit to use all the particles. use a value for n_fit between 0 and 1 to use a certain percentage
    # (5, 1, 10_000),
    # (5, 2, 10_000),
    # (5, 3, 10_000),
    # (5, 4, 10_000),
    # (6, 1, 10_000),
    # (6, 2, 10_000),
    # (6, 3, 10_000),
    # (6, 4, 10_000),
]
calculate_these.extend(
    [
        (fn, bw_kernel_index, n_fit)
        for fn in (5,6)
        for bw_kernel_index in (3,4)
        for n_fit in (
            500_000,
        )
    ]
)

# calculate_these.extend(
#     [
#         (fn, bw_kernel_index, n_fit)
#         for fn in (5,6)
#         for bw_kernel_index in (1,2)
#         for n_fit in (
#             1_000,
#             5_000,
#             10_000,
#             20_000,
#             50_000,
#             100_000,
#             200_000,
#             500_000,
#             1_000_000,
#             2_000_000,
#         )
#     ]
# )

calculate_these

### Create partial data files to use for fitting

In [ ]:
copy_data_folder = "./copied_data"

from pathlib import Path
copy_data_folder = Path(copy_data_folder).absolute()
if not copy_data_folder.exists():
    Path.mkdir(copy_data_folder)

In [ ]:
import numpy as np

copy_files = {(int(fn), int(n_fit)): str(p) for fn, n_fit, p in [str(p.with_suffix("").stem).split("_")+[p] for p in copy_data_folder.glob("*.mcpl.gz")]}
print(copy_files)

for (fn, bw_kernel_num, n_fit) in calculate_these:
    file = example_paths[fn]
    copy_file = str(copy_data_folder / Path(str(fn)+"_"+str(n_fit)+".mcpl.gz"))
    fn_n_fit = (fn, n_fit)
    if fn_n_fit in copy_files.keys():
        continue
    copy_data = read_data(file_path=file, n=n_fit)
    kds.write_mcpl(copy_file,
                    ekin=copy_data["ekin"],
                    x=copy_data["x"],
                    y=copy_data["y"],
                    z=copy_data["z"],
                    ux=copy_data["ux"],
                    uy=copy_data["uy"],
                    uz=copy_data["uz"],
                    time=copy_data["time"],
                    weight=copy_data["weight"][0] if is_constant_column(copy_data["weight"]) else copy_data["weight"],
                    pdgcode=copy_data["pdgcode"][0] if is_constant_column(copy_data["pdgcode"]) else copy_data["pdgcode"],
                    polx=None,
                    poly=None,
                    polz=None,
                    userflags=None,
                    )
    copy_files[fn_n_fit] = copy_file


### run calculations, to obtain model xml files

In [ ]:
models_folder="./calculated_models"

from pathlib import Path
models_folder = Path(models_folder).absolute()
if not models_folder.exists():
    Path.mkdir(models_folder)

In [ ]:
import math
import time
import numpy as np

for i, (fn, bw_kernel_num, n_fit) in enumerate(calculate_these):
    
    fn_n_fit = (fn, n_fit)
    plist = kds.PList(
        filename=copy_files[fn_n_fit],
        readformat="mcpl",
        pt=pt[fn] if fn in pt.keys() else "n",
        trasl=trasl[fn] if fn in trasl.keys() else None,
        rot=rot[fn] if fn in rot.keys() else None,
        switch_x2z=switch_x2z[fn] if fn in switch_x2z.keys() else False,
        set_params=True)

    N = plist.N

    bw, kernel = bw_kernel_combinations[bw_kernel_num]
    if n_fit == -1: n_fit = N
    elif 0 < n_fit < 1: n_fit = math.floor(N * n_fit)
    else: n_fit = max(min(N, n_fit), 0)
    if n_fit == 0:
        print("n_fit became zero, calculation will be skipped now.")
        continue

    source = kds.KDSource(
            plist=plist,
            geom=geometries[fn],
            bw=bw,
            kernel=kernel)
    
    fit_success = False
    start = time.perf_counter()
    mlcv_grid_multiplier = 5
    mlcv_grid_start = (-0.1 if n_fit < 100_000 else -0.5) if n_fit < 500_000 else -2.5
    mlcv_grid_stop = (0.1 if n_fit < 100_000 else 0.5) if n_fit < 500_000 else 2.5
    mlcv_grid_num = 8
    mlcv_attempt = 1    
    while fit_success == False and mlcv_attempt <= 3:
        try:
            if bw=="mlcv":
                grid = np.logspace(mlcv_grid_start, mlcv_grid_stop, mlcv_grid_num)
                print("attempting grid logspace start stop", str(mlcv_grid_start), str(mlcv_grid_stop))
                source.fit(N=n_fit, grid=grid)
            else:
                source.fit(N=n_fit)
        except:
            print("Fitting failed")
            if bw=="mlcv":
                mlcv_grid_start *= mlcv_grid_multiplier
                mlcv_grid_stop *= mlcv_grid_multiplier
        else:
            fit_success = True
        finally:
            if bw=="mlcv":
                mlcv_attempt += 1
            else:
                break        
    end = time.perf_counter()
    elapsed_seconds = int(end-start)
    print(f"Elapsed: {elapsed_seconds} s")

    if fit_success:
        xmlfilename = str(models_folder)+"/file_"+str(fn)+"_"+bw+"_"+kernel+"_"+str(N)+"_"+str(n_fit)+"_"+str(elapsed_seconds)+"s.xml"
        source.save(xmlfilename=xmlfilename)


## generate (resample) data from the models (skip)

In [4]:
gen_data_folder = "./generated_data"
gen_N = 2_000_000 # number of particles to generate per model
RNG = 123456789

from pathlib import Path

gen_data_folder = Path(gen_data_folder).absolute()
if not gen_data_folder.exists():
    Path.mkdir(gen_data_folder)

In [ ]:
for model_path in models_folder.glob("*"):
    #print(model_path)
    stem = model_path.with_suffix("").stem
    f, fn, bw, kernel, N, n_fit, elapsed = stem.split(sep="_")
    # print(f, fn, bw, kernel, N, n_fit, elapsed)
    fn = int(fn)
    N = int(N)
    n_fit = int(n_fit)
    dest_path = gen_data_folder / Path(stem).with_suffix(".mcpl.gz")
    # print(dest_path)
    if dest_path in gen_data_folder.glob("*"):
        print("skipping", dest_path)
        continue
    resample_df(source=model_path, dest=dest_path, nout=gen_N, rng=RNG, force=True, no_df=True, del_dest=False, del_source=False)

## read data and produce metrics (skip)

In [5]:
from collections import defaultdict
def nested_dict():
    return defaultdict(nested_dict)
def normalize_nested_dict(d):
    if isinstance(d, defaultdict):
        return {k: normalize_nested_dict(v) for k, v in d.items()}
    return d

In [6]:
from brightify.data_comparison import kolmogorov_smirnov_test, wasserstein, population_stability_index, kl_divergence, js_distance
from brightify.data_comparison import energy_distance, maximum_mean_discrepancy, c2st
import time

original_data_metrics = nested_dict()
gen_data_metrics = nested_dict()
gen_data_files = sorted(gen_data_folder.glob("*"))
for i_data, gen_data_path in enumerate(gen_data_files):
    print("looking at file", str(i_data+1), "of", str(len(gen_data_files)), ":", str(gen_data_path))
    stem = gen_data_path.with_suffix("").stem
    f, fn, bw, kernel, N, n_fit, elapsed = stem.split(sep="_")
    # print(elapsed)
    fn = int(fn)

    orig_data = read_data(example_paths[fn])
    if not fn in original_data_metrics:
        orig_summary = orig_data.describe().loc[['mean', 'std', 'min', 'max']]
        for c in orig_summary.columns:
            original_data_metrics[fn]['mean'][c] = orig_summary.loc['mean', c]
            original_data_metrics[fn]['std'][c] = orig_summary.loc['std', c]
            original_data_metrics[fn]['min'][c] = orig_summary.loc['min', c]
            original_data_metrics[fn]['max'][c] = orig_summary.loc['max', c]

    bw_kernel = (bw, kernel)
    n_fit = int(n_fit)
    elapsed = int(elapsed.rstrip("s"))
    gen_data_metrics[fn][bw_kernel][n_fit]['elapsed'] = elapsed
    gen_data = read_data(gen_data_path)
    gen_summary = gen_data.describe().loc[['mean', 'std', 'min', 'max']]
    for c in gen_summary.columns:
        gen_data_metrics[fn][bw_kernel][n_fit]['mean'][c] = gen_summary.loc['mean', c]
        gen_data_metrics[fn][bw_kernel][n_fit]['std'][c] = gen_summary.loc['std', c]
        gen_data_metrics[fn][bw_kernel][n_fit]['min'][c] = gen_summary.loc['min', c]
        gen_data_metrics[fn][bw_kernel][n_fit]['max'][c] = gen_summary.loc['max', c]  
    
    columns = ["ekin", "x", "y", "z", "ux", "uy", "uz"]

    # univariate:
    try:
        print("kolmogorov_smirnov_test")
        start_time = time.perf_counter()
        result = kolmogorov_smirnov_test(gen_data, orig_data, columns=columns)
        for c, v in result.items():
            gen_data_metrics[fn][bw_kernel][n_fit]['kolmogorov_smirnov_test'][c] = v
    except:
        print("FAIL: kolmogorov_smirnov_test", fn, bw_kernel, n_fit)
    finally:
        print("elapsed (s):", int(time.perf_counter() - start_time))
    try:
        print("wasserstein")
        start_time = time.perf_counter()
        result = wasserstein(gen_data, orig_data, columns=columns)
        for c, v in result.items():
            gen_data_metrics[fn][bw_kernel][n_fit]['wasserstein'][c] = v
    except:
        print("FAIL: wasserstein", fn, bw_kernel, n_fit)
    finally:
        print("elapsed (s):", int(time.perf_counter() - start_time))
    try:
        print("population_stability_index")
        start_time = time.perf_counter()
        result = population_stability_index(gen_data, orig_data, columns=columns)
        for c, v in result.items():
            gen_data_metrics[fn][bw_kernel][n_fit]['population_stability_index'][c] = v
    except:
        print("FAIL: population_stability_index", fn, bw_kernel, n_fit)
    finally:
        print("elapsed (s):", int(time.perf_counter() - start_time))
    try:
        print("kl_divergence")
        start_time = time.perf_counter()
        result = kl_divergence(df_p=orig_data, df_q=gen_data, columns=columns)
        for c, v in result.items():
            gen_data_metrics[fn][bw_kernel][n_fit]['kl_divergence'][c] = v
    except:
        print("FAIL: kl_divergence", fn, bw_kernel, n_fit)
    finally:
        print("elapsed (s):", int(time.perf_counter() - start_time))
    try:
        print("js_distance")
        start_time = time.perf_counter()
        result = js_distance(df_p=orig_data, df_q=gen_data, columns=columns)
        for c, v in result.items():
            gen_data_metrics[fn][bw_kernel][n_fit]['js_distance'][c] = v
    except:
        print("FAIL: js_distance", fn, bw_kernel, n_fit)
    finally:
        print("elapsed (s):", int(time.perf_counter() - start_time))

    # multivariate:
    try:
        print("energy_distance")
        start_time = time.perf_counter()
        result = energy_distance(orig_data, gen_data, columns=columns)
        gen_data_metrics[fn][bw_kernel][n_fit]['energy_distance'] = result
    except:
        print("FAIL: energy_distance", fn, bw_kernel, n_fit)
    finally:
        print("elapsed (s):", int(time.perf_counter() - start_time))
    try:
        print("maximum_mean_discrepancy")
        start_time = time.perf_counter()
        result = maximum_mean_discrepancy(orig_data, gen_data, columns=columns)
        gen_data_metrics[fn][bw_kernel][n_fit]['maximum_mean_discrepancy'] = result
    except:
        print("FAIL: maximum_mean_discrepancy", fn, bw_kernel, n_fit)
    finally:
        print("elapsed (s):", int(time.perf_counter() - start_time))
    try:
        print("c2st")
        start_time = time.perf_counter()
        result = c2st(orig_data, gen_data, columns=columns)
        gen_data_metrics[fn][bw_kernel][n_fit]['c2st'] = result
    except:
        print("FAIL: c2st", fn, bw_kernel, n_fit)
    finally:
        print("elapsed (s):", int(time.perf_counter() - start_time))

original_data_metrics = normalize_nested_dict(original_data_metrics)
gen_data_metrics = normalize_nested_dict(gen_data_metrics)

looking at file 1 of 70 : /home/jhonal/Brightify/generated_data/file_5_mlcv_epa_100000_100000_1729s.mcpl.gz
kolmogorov_smirnov_test
elapsed (s): 0
wasserstein
elapsed (s): 0
population_stability_index
elapsed (s): 1
kl_divergence
elapsed (s): 1
js_distance
elapsed (s): 1
energy_distance
elapsed (s): 9
maximum_mean_discrepancy


: 

### 1d (marginal) histograms:

In [ ]:
bin_count = 100
bin_extra = 10 # amount of extra bins left and right of min and max found in original data (to account for generated data values outside of it)

bins_min = nested_dict()
bins_max = nested_dict()
for fn, gdm_fn in gen_data_metrics.items():
    for c, v in original_data_metrics[fn]['max'].items():
        if not c in bins_max[fn] or bins_max[fn][c] < v:
            bins_max[fn][c] = v
    for c, v in original_data_metrics[fn]['min'].items():
        if not c in bins_min[fn] or bins_min[fn][c] > v:
            bins_min[fn][c] = v

bins = nested_dict()
orig_hists = nested_dict()
for fn in bins_min.keys():
    for c in bins_min[fn].keys():
        ma = bins_max[fn][c]
        mi = bins_min[fn][c]
        span = ma - mi
        per_bin = span/bin_count
        extra_span = per_bin * bin_extra
        ma += extra_span
        mi -= extra_span
        bins[fn][c] = np.linspace(mi, ma, bin_count + 2*bin_extra + 1)
    original_path = example_paths[fn]
    data = read_data(original_path)
    for c, b in bins[fn].items():
        orig_hists[fn][c] = np.histogram(data[c], bins=b)
bins = normalize_nested_dict(bins)
orig_hists = normalize_nested_dict(orig_hists)

gen_hists = nested_dict()
for gen_data_path in gen_data_folder.glob("*"):
    stem = gen_data_path.with_suffix("").stem
    f, fn, bw, kernel, N, n_fit, elapsed = stem.split(sep="_")
    # print(elapsed)
    fn = int(fn)
    bw_kernel = (bw, kernel)
    n_fit = int(n_fit)
    gen_data = read_data(gen_data_path)
    for c in bins[fn].keys():
        gen_hists[fn][bw_kernel][n_fit][c] = np.histogram(gen_data[c], bins[fn][c])
gen_hists = normalize_nested_dict(gen_hists)

In [ ]:
gen_hists

### 2d histograms:

In [ ]:
from itertools import combinations

# 1. Compute 2D histograms for original data
orig_hists_2d = nested_dict()

for fn in bins.keys():
    original_path = example_paths[fn]
    data = read_data(original_path)
    dims = list(bins[fn].keys())
    
    # combinations(dims, 2) generates all unique pairs without duplicates or self-pairs
    for dim_x, dim_y in combinations(dims, 2):
        counts_2d, edges_x, edges_y = np.histogram2d(
            data[dim_x],
            data[dim_y],
            bins=[bins[fn][dim_x], bins[fn][dim_y]]
        )
        orig_hists_2d[fn][(dim_x, dim_y)] = (counts_2d, edges_x, edges_y)

orig_hists_2d = normalize_nested_dict(orig_hists_2d)

# 2. Compute 2D histograms for generated data
gen_hists_2d = nested_dict()

for gen_data_path in gen_data_folder.glob("*"):
    stem = gen_data_path.with_suffix("").stem
    f, fn, bw, kernel, N, n_fit, elapsed = stem.split(sep="_")
    fn = int(fn)
    bw_kernel = (bw, kernel)
    n_fit = int(n_fit)
    
    gen_data = read_data(gen_data_path)
    dims = list(bins[fn].keys())
    
    for dim_x, dim_y in combinations(dims, 2):
        counts_2d, edges_x, edges_y = np.histogram2d(
            gen_data[dim_x],
            gen_data[dim_y],
            bins=[bins[fn][dim_x], bins[fn][dim_y]]
        )
        gen_hists_2d[fn][bw_kernel][n_fit][(dim_x, dim_y)] = (counts_2d, edges_x, edges_y)

gen_hists_2d = normalize_nested_dict(gen_hists_2d)

## save / load

In [ ]:
results_dir = "./pickled_results"

import pickle as pkl
from pathlib import Path
results_dir = Path(results_dir).absolute()
    
gen_data_file = str(results_dir / "gen_data.pkl")
orig_data_file = str(results_dir / "orig_data.pkl")
gen_hists_file = str(results_dir / "gen_hists.pkl")
orig_hists_file = str(results_dir / "orig_hists.pkl")
gen_hists_2d_file = str(results_dir / "gen_hists_2d.pkl")
orig_hists_2d_file = str(results_dir / "orig_hists_2d.pkl")

#### save pickle (skip)

In [ ]:
if not results_dir.exists():
    Path.mkdir(results_dir)

with open(gen_data_file, "wb") as f:
    pkl.dump(gen_data_metrics, f)
with open(orig_data_file, "wb") as f:
    pkl.dump(original_data_metrics, f)
with open(gen_hists_file, "wb") as f:
    pkl.dump(gen_hists, f)
with open(orig_hists_file, "wb") as f:
    pkl.dump(orig_hists, f)

import os

chunk_size = 50 * 1024 * 1024  # 50 MB per chunk (well under GitHub's 100 MB limit)
data_bytes = pkl.dumps(gen_hists_2d)

for i, start in enumerate(range(0, len(data_bytes), chunk_size)):
    chunk_file = f"{gen_hists_2d_file}.part{i}"
    with open(chunk_file, "wb") as f:
        f.write(data_bytes[start : start + chunk_size])
    
with open(orig_hists_2d_file, "wb") as f:
    pkl.dump(orig_hists_2d, f)

### load pickle

In [ ]:
with open(gen_data_file, "rb") as f:
    gen_data_metrics = pkl.load(f)
with open(orig_data_file, "rb") as f:
    original_data_metrics = pkl.load(f)
with open(gen_hists_file, "rb") as f:
    gen_hists = pkl.load(f)
with open(orig_hists_file, "rb") as f:
    orig_hists = pkl.load(f)

import glob

# Find all parts and sort them numerically by part number
chunk_files = sorted(
    glob.glob(f"{gen_hists_2d_file}.part*"),
    key=lambda x: int(x.split(".part")[-1])
)

data_bytes = bytearray()
for chunk_file in chunk_files:
    with open(chunk_file, "rb") as f:
        data_bytes.extend(f.read())

gen_hists_2d = pkl.loads(data_bytes)

with open(orig_hists_2d_file, "rb") as f:
    orig_hists_2d = pkl.load(f)

In [ ]:
# gen_data_metrics

In [ ]:
# original_data_metrics

In [ ]:
gen_hists

In [ ]:
orig_hists

## Visualize metrics

### 1d graphs and histograms:

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

file_widget = widgets.Dropdown(options=list(gen_data_metrics.keys()), description='file no.:')
prop_widget = widgets.Dropdown(options=["mean", "std", "min", "max", "wasserstein", "kl_div", "js_dist", "c2st", "hist", "elapsed"], description='property:')
dim_widget = widgets.Dropdown(options=["ekin", "x", "y", "z", "ux", "uy", "uz", "weight", "pdgcode"], description='dimension:')

plot_button = widgets.Button(description="Generate Plot", button_style='primary')

out = widgets.Output()

_is_running = False

def on_button_clicked(b):
    global _is_running
    if _is_running:
        return

    _is_running = True
    b.description = "Plotting..."
    b.disabled = True

    try:
        with out:
            clear_output(wait=True)

            file = file_widget.value
            prop = prop_widget.value
            dimension = dim_widget.value

            gen_metrics = gen_data_metrics[file]
            orig_metrics = original_data_metrics[file]
            bw_kernels = list(gen_metrics.keys())
            bws = sorted(list({bw_kernel[0] for bw_kernel in bw_kernels}))
            kernels = sorted(list({bw_kernel[1] for bw_kernel in bw_kernels}))
            # print(bws, kernels)
            n_bws, n_kernels = len(bws), len(kernels)
            # print(n_bws, n_kernels)
            fig, axs = plt.subplots(n_bws, n_kernels, figsize=(6*n_kernels, 4*n_bws), squeeze=False, constrained_layout=True)
            if not prop=="hist":
                fig2, ax2 = plt.subplots(figsize=(6*n_kernels, 4*n_bws), constrained_layout=True)              
            ylabel=""
            ax2_hline_orig_value_has_been_plotted = False
            for i_bw, bw in enumerate(bws):
                for i_kernel, kernel in enumerate(kernels):
                    ax = axs[i_bw, i_kernel]
                    bw_kernel = (bw, kernel)
                    if prop == "hist":
                        gen_h_bw = gen_hists[file][bw_kernel]
                        for n_fit, gen_h_n in sorted(gen_h_bw.items(), key=lambda tup:tup[0]):
                            counts, edges = gen_h_n[dimension]
                            norm_counts = counts/np.sum(counts)
                            ax.stairs(norm_counts, edges, label="n_fit="+str(n_fit)) 
                        counts, edges = orig_hists[file][dimension]
                        norm_counts = counts/np.sum(counts)
                        ax.stairs(norm_counts, edges, label="orig. data", linestyle="--", color='black')            
                        ax.set_xlabel(dimension+" histogram")
                        ax.set_ylabel("counts (normalized)")
                        ax.legend()
                        ax.grid()
                    else:                    
                        metrics = gen_metrics[bw_kernel]
                        n_fits = sorted(list(metrics.keys()))
                        # print(n_fits)
                        ylabel = prop
                        if prop == "elapsed":
                            prop_values = [metrics[n_fit][prop] for n_fit in n_fits]
                            # print(prop_values)
                            ylabel += " (s)"
                            ax.set_yscale("symlog")
                            if i_bw == 0 and i_kernel == 0: ax2.set_yscale("symlog")
                        elif prop == "c2st":
                            prop_values = [metrics[n_fit][prop] for n_fit in n_fits]
                            ylabel += " (multi-var)"
                        else:
                            prop_values = [metrics[n_fit][prop].get(dimension, np.nan) for n_fit in n_fits]
                            ylabel += " "+dimension
                            orig_metr_prop = orig_metrics.get(prop, None)
                            if orig_metr_prop:
                                orig_value = orig_metr_prop[dimension]
                                ax.axhline(orig_value, color='gray', linestyle='--', label="orig. data")
                                if not ax2_hline_orig_value_has_been_plotted:
                                    ax2.axhline(orig_value, color='gray', linestyle='--', label="orig. data")
                                    ax2_hline_orig_value_has_been_plotted = True
                        ax.plot(n_fits, prop_values, marker='o', linestyle="-.", label="gen. data")
                        ax2.plot(n_fits, prop_values, marker='o', linestyle="-.", label="gen. data "+bw+" "+kernel)
                        ax.set_xlabel("n_fit")
                        ax.set_ylabel(ylabel)
                        ax.set_xscale("log")
                        ax.set_xscale("log")
                    ax.set_title(bw+" "+kernel)
                    ax.legend()
                    ax.grid()
            fig.suptitle("file no. "+str(file))
            if not prop=="hist":
                ax2.set_xlabel("n_fit")
                ax2.set_ylabel(ylabel)
                ax2.legend()
                ax2.grid()          
            plt.show()
            plt.close(fig)
            if not prop=="hist":
                plt.close(fig2)
            return
    finally:
        # Always release the lock and restore the button even if something errors
        _is_running = False
        b.description = "Generate Plot"
        b.disabled = False

# Link the button click to the function
plot_button._click_handlers.callbacks.clear()
plot_button.on_click(on_button_clicked)

# Display the UI block
display(widgets.VBox([file_widget, prop_widget, dim_widget, plot_button, out]))

### 2d histograms:

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np

# Widgets
file_2d_widget = widgets.Dropdown(options=list(gen_hists_2d.keys()), description='file no.:')
dim1_widget = widgets.Dropdown(
    options=["ekin", "x", "y", "z", "ux", "uy", "uz", "weight", "pdgcode"],
    value="x",
    description='dim 1 (X):'
)
dim2_widget = widgets.Dropdown(
    options=["ekin", "x", "y", "z", "ux", "uy", "uz", "weight", "pdgcode"],
    value="ux",
    description='dim 2 (Y):'
)
n_fit_widget = widgets.Dropdown(description='n_fit:')
auto_zoom_widget = widgets.Checkbox(value=True, description='Auto-zoom to data')
log_scale_widget = widgets.Checkbox(value=True, description='Log color scale')
show_contours_widget = widgets.Checkbox(value=True, description='Orig. contours')

plot_2d_button = widgets.Button(description="Generate 2D Hist", button_style='primary')
out_2d = widgets.Output()

def update_n_fit_options(*args):
    """Collect ALL unique n_fit values across ALL (bw, kernel) pairs for the selected file."""
    file = file_2d_widget.value
    if file not in gen_hists_2d:
        n_fit_widget.options = []
        return
    
    all_n_fits = set()
    for bw_kernel_dict in gen_hists_2d[file].values():
        if isinstance(bw_kernel_dict, dict):
            all_n_fits.update(bw_kernel_dict.keys())
            
    n_fits = sorted(list(all_n_fits))
    n_fit_widget.options = n_fits
    if n_fits:
        n_fit_widget.value = n_fits[-1]

file_2d_widget.observe(update_n_fit_options, names='value')
update_n_fit_options()

def get_2d_hist(hist_dict_2d, dim_x, dim_y):
    """Safely retrieves normalized 2D density and fixes zero-width constant dimensions."""
    if not hist_dict_2d:
        return None, None, None
    if (dim_x, dim_y) in hist_dict_2d:
        counts, ex, ey = hist_dict_2d[(dim_x, dim_y)]
        grid = counts.T
    elif (dim_y, dim_x) in hist_dict_2d:
        counts, ey, ex = hist_dict_2d[(dim_y, dim_x)]
        grid = counts
    else:
        return None, None, None

    total = np.sum(grid)
    norm_grid = grid / total if total > 0 else grid

    # Fix collapsed edges if a dimension was constant (e.g. pdgcode)
    if ex[0] == ex[-1]:
        val = ex[0]
        norm_grid = np.sum(norm_grid, axis=1, keepdims=True)
        ex = np.array([val - 0.5, val + 0.5])
    if ey[0] == ey[-1]:
        val = ey[0]
        norm_grid = np.sum(norm_grid, axis=0, keepdims=True)
        ey = np.array([val - 0.5, val + 0.5])

    return norm_grid, ex, ey

def get_tight_bounds(hist_2d, ex, ey, pad_bins=2):
    """Finds x and y axis limits tightly framing non-zero bins, safe for constant axes."""
    nonzero_y, nonzero_x = np.nonzero(hist_2d > 0)
    if len(nonzero_x) == 0 or len(nonzero_y) == 0:
        x_lo, x_hi = ex[0], ex[-1]
        y_lo, y_hi = ey[0], ey[-1]
    else:
        x_min_idx = max(0, nonzero_x.min() - pad_bins)
        x_max_idx = min(len(ex) - 1, nonzero_x.max() + 1 + pad_bins)
        y_min_idx = max(0, nonzero_y.min() - pad_bins)
        y_max_idx = min(len(ey) - 1, nonzero_y.max() + 1 + pad_bins)
        x_lo, x_hi = ex[x_min_idx], ex[x_max_idx]
        y_lo, y_hi = ey[y_min_idx], ey[y_max_idx]

    if x_lo == x_hi:
        x_lo, x_hi = x_lo - 0.5, x_hi + 0.5
    if y_lo == y_hi:
        y_lo, y_hi = y_lo - 0.5, y_hi + 0.5

    return (x_lo, x_hi), (y_lo, y_hi)

_is_running_2d = False

def on_2d_button_clicked(b):
    global _is_running_2d
    if _is_running_2d:
        return

    _is_running_2d = True
    b.description = "Plotting..."
    b.disabled = True

    try:
        with out_2d:
            clear_output(wait=True)
            plt.close('all')  # Prevent stale unclosed figures from accumulating in VS Code

            file = file_2d_widget.value
            dim_x = dim1_widget.value
            dim_y = dim2_widget.value
            n_fit = n_fit_widget.value
            auto_zoom = auto_zoom_widget.value
            use_log = log_scale_widget.value
            overlay_contours = show_contours_widget.value

            if dim_x == dim_y:
                print("Please select two different dimensions for X and Y.")
                return

            # 1. Safely load Original 2D Histogram
            orig_file_dict = orig_hists_2d.get(file, {})
            orig_2d, orig_ex, orig_ey = get_2d_hist(orig_file_dict, dim_x, dim_y)
            if orig_2d is None:
                print(f"No original 2D histogram found for ({dim_x}, {dim_y}) in file {file}.")
                return

            # Compute tight zoom limits based on original data (or combined data)
            xlim, ylim = get_tight_bounds(orig_2d, orig_ex, orig_ey) if auto_zoom else ((orig_ex[0], orig_ex[-1]), (orig_ey[0], orig_ey[-1]))

            # Shared color normalization
            positive_vals = orig_2d[orig_2d > 0]
            vmax = np.max(orig_2d) if len(positive_vals) > 0 else 1.0
            vmin = max(np.min(positive_vals), vmax * 1e-4) if (use_log and len(positive_vals) > 0) else 0.0
            norm = LogNorm(vmin=vmin, vmax=vmax) if use_log else None

            # 2. Plot Original Data Reference
            fig_orig, ax_orig = plt.subplots(figsize=(5.5, 4.2), constrained_layout=True)
            mesh_o = ax_orig.pcolormesh(
                orig_ex, orig_ey, orig_2d, shading='flat', cmap='viridis',
                norm=norm, vmin=None if use_log else 0, vmax=None if use_log else vmax
            )
            fig_orig.colorbar(mesh_o, ax=ax_orig, label="Normalized Density")
            ax_orig.set_xlim(xlim)
            ax_orig.set_ylim(ylim)
            ax_orig.set_xlabel(dim_x)
            ax_orig.set_ylabel(dim_y)
            ax_orig.set_title(f"ORIGINAL DATA: {dim_x} vs {dim_y} (file {file})")
            plt.show()
            plt.close(fig_orig)

            # 3. Plot Generated Data Grid safely
            file_gen_dict = gen_hists_2d.get(file, {})
            bw_kernels = list(file_gen_dict.keys())
            if not bw_kernels:
                print(f"No generated histograms found for file {file}.")
                return

            bws = sorted(list({bk[0] for bk in bw_kernels}))
            kernels = sorted(list({bk[1] for bk in bw_kernels}))
            n_bws, n_kernels = len(bws), len(kernels)

            fig, axs = plt.subplots(
                n_bws, n_kernels,
                figsize=(5.5 * n_kernels, 4.5 * n_bws),
                squeeze=False,
                constrained_layout=True
            )

            orig_cx = 0.5 * (orig_ex[:-1] + orig_ex[1:])
            orig_cy = 0.5 * (orig_ey[:-1] + orig_ey[1:])
            contour_levels = np.geomspace(vmin * 5, vmax * 0.7, 4) if use_log else np.linspace(0.1 * vmax, 0.8 * vmax, 4)

            for i_bw, bw in enumerate(bws):
                for i_kernel, kernel in enumerate(kernels):
                    ax = axs[i_bw, i_kernel]
                    bw_kernel = (bw, kernel)

                    # Safe nested lookup
                    gen_dict = file_gen_dict.get(bw_kernel, {}).get(n_fit, None)
                    gen_2d, ex, ey = get_2d_hist(gen_dict, dim_x, dim_y)

                    if gen_2d is None:
                        ax.text(0.5, 0.5, f"Missing data\n({bw}, {kernel})\nn_fit={n_fit}",
                                ha='center', va='center', transform=ax.transAxes, color='gray')
                        ax.set_xticks([])
                        ax.set_yticks([])
                        ax.set_title(f"{bw} | {kernel} (N/A)")
                        continue

                    mesh = ax.pcolormesh(
                        ex, ey, gen_2d, shading='flat', cmap='viridis',
                        norm=norm, vmin=None if use_log else 0, vmax=None if use_log else vmax
                    )
                    fig.colorbar(mesh, ax=ax, label="Normalized Density")

                    if overlay_contours and orig_2d.shape[0] >= 2 and orig_2d.shape[1] >= 2:
                        try:
                            ax.contour(
                                orig_cx, orig_cy, orig_2d,
                                levels=contour_levels,
                                colors='white',
                                linewidths=1.0,
                                alpha=0.85
                            )
                        except (ValueError, TypeError):
                            pass  # Skip contours if levels aren't strictly increasing (e.g. flat distribution)

                    ax.set_xlim(xlim)
                    ax.set_ylim(ylim)
                    ax.set_xlabel(dim_x)
                    ax.set_ylabel(dim_y)
                    ax.set_title(f"{bw} | {kernel} (n_fit={n_fit})")

            fig.suptitle(f"Generated 2D Histograms: {dim_x} vs {dim_y} (n_fit={n_fit})")
            plt.show()
            plt.close(fig)
    finally:
        plt.close('all')  # Guarantees no ghost figures are ever leaked
        _is_running_2d = False
        b.description = "Generate 2D Hist"
        b.disabled = False

plot_2d_button._click_handlers.callbacks.clear()
plot_2d_button.on_click(on_2d_button_clicked)

ui_2d = widgets.VBox([
    widgets.HBox([file_2d_widget, n_fit_widget]),
    widgets.HBox([dim1_widget, dim2_widget]),
    widgets.HBox([auto_zoom_widget, log_scale_widget, show_contours_widget]),
    plot_2d_button,
    out_2d
])
display(ui_2d)

### Observations

1. The silverman method is incredibly fast compared to mlcv, whose run time scales approximately as n_fit^2 (quadratic with the amount of particles used for fitting).
2. Regarding the mean values, silverman vs. mlcv difference can not be seen very well. For file 5, at n_fit =~ 10^4, the mean of x and y meet the original data mean for the first time and then touch / cross it, getting sort of closer to it with increasing n_fit. While for file 6, x and y they come close to the true average at around n_fit = 10^5, and before that cross through it, showing that mere 1000 particles are definitely insufficient. All that independently of whether silverman of mlcv is looked at.
3. Between gaussian and epa kernel, no large difference is observable, except that the variance/std of the values is a bit smaller for the Epanechnikov kernel.
